## Prompt 1

 write a query in the notebook that can find the availability (number of available vavancies) for the particular tour_id on the particular tour_date in the future

 ## Prompt 2 

I must the availability for the particular tour point in time correct for the past dates and for the future date
 

## Prompt 3

Write an sql code in the new empty cell that finds availability (vacancies) of the tour_id that happens on the tour_date on the booking_date (the date when a costumer would make a reservation)

I need to find bookings of the particular 'tour_id' happening on the 'tour_date' that were effective on the 'date' in the past (point in time correct) before the 'tour_date'. Which table can I use for that? 

write an sql code in the new empty cell that finds availability (vacancies) of the tour_id that happens on the tour_date on the booking_date (the date when a costumer would make a reservation)

In [17]:
spark

In [3]:
top_tours_sql = """
WITH top_tours AS (
    SELECT
        tour_id,
        COUNT(booking_id)  AS booking_count,
        SUM(tickets)       AS total_tickets,
        ROUND(SUM(gmv), 2) AS total_gmv_eur
    FROM production.dwh.fact_booking
    WHERE
        YEAR(date_of_checkout) = 2024
        AND is_fraud = false
        AND date_of_cancelation IS NULL
    GROUP BY tour_id
    ORDER BY booking_count DESC
    LIMIT 10
)

SELECT
    t.tour_id,
    t.booking_count,
    t.total_tickets,
    t.total_gmv_eur,
    b.tour_option_id
FROM top_tours t
JOIN production.dwh.fact_booking b
    ON  b.tour_id = t.tour_id
    AND YEAR(b.date_of_checkout) = 2024
    AND b.is_fraud = false
    AND b.date_of_cancelation IS NULL
GROUP BY
    t.tour_id,
    t.booking_count,
    t.total_tickets,
    t.total_gmv_eur,
    b.tour_option_id
ORDER BY t.booking_count DESC, t.tour_id, b.tour_option_id
"""

df_top_tours = spark.sql(top_tours_sql)

In [4]:
display(df_top_tours)

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737
3,50027,121163,311630,10416319.95,654640
4,50027,121163,311630,10416319.95,654660
5,50027,121163,311630,10416319.95,741729
6,53791,107484,271705,3435836.97,74290
7,193940,102208,278280,4338708.96,300869
8,73250,98289,249158,3220068.00,376779
9,145779,81545,172828,4345691.44,218301


In [5]:
df_top_tours.limit(3).toPandas()

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737


In [6]:
df_top_tours.select("tour_id", "booking_count", "total_tickets", "total_gmv_eur").distinct().orderBy("booking_count", ascending=False).limit(3).toPandas()

,tour_id,booking_count,total_tickets,total_gmv_eur
0,62214,128680,362286,11852559.98
1,50027,121163,311630,10416319.95
2,53791,107484,271705,3435836.97


In [7]:
spark.sql("SHOW TABLES IN production.dwh LIKE '*booking*'").show(50, truncate=False)

+--------+---------------------------------+-----------+
|database|tableName                        |isTemporary|
+--------+---------------------------------+-----------+
|dwh     |agg_booking_gwc                  |false      |
|dwh     |agg_bookings_checkout_date_daily |false      |
|dwh     |agg_bookings_checkout_date_hourly|false      |
|dwh     |agg_bookings_tour_checkout_daily |false      |
|dwh     |agg_bookings_tour_travel_daily   |false      |
|dwh     |agg_bookings_travel_date_daily   |false      |
|dwh     |bookings_change_monitoring       |false      |
|dwh     |dim_booking_addon_details        |false      |
|dwh     |dim_booking_cancellation_category|false      |
|dwh     |dim_booking_cancellation_reason  |false      |
|dwh     |dim_booking_cancellation_user    |false      |
|dwh     |dim_booking_cohort               |false      |
|dwh     |dim_booking_geo_mapping          |false      |
|dwh     |dim_booking_history              |false      |
|dwh     |dim_booking_language 

In [8]:
spark.sql("DESCRIBE production.dwh.fact_booking_snapshot").show(50, truncate=False)

+----------------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                                |data_type    |comment                                                                                                                                                 |
+----------------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_timestamp                      |timestamp    |The 'snapshot_timestamp' column records the exact time when the snapshot of the booking data was taken. (AI-generated field)                            |
|booking_id                              |bigint       |The 'booking_id' column uniquely identifies each booking within the system. (AI-generated field)        

In [ ]:
spark.sql("""
    SELECT DATE(snapshot_timestamp) AS snapshot_date, COUNT(*) AS rows
    FROM production.dwh.fact_booking_snapshot
    WHERE snapshot_timestamp >= '2026-06-01'
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 10
""").show()

In [34]:
from pyspark.sql import functions as F

# ── params ────────────────────────────────────────────────────────────────────
TOUR_ID          = 12345                  # change me
TOUR_DATE        = "2026-07-15"           # change me  (YYYY-MM-DD)
# Point-in-time observation: use current_timestamp() for live/future availability,
# or a specific past timestamp to reconstruct historical availability.
AS_OF_TIMESTAMP  = F.current_timestamp()  # e.g. F.lit("2025-07-14 10:00:00")
# ─────────────────────────────────────────────────────────────────────────────

df = (
    spark.table("production.supply.fact_available_time_slot_history_compacted")
    .filter(
        (F.col("tour_id") == TOUR_ID) &
        (F.to_date("date_time_local") == F.lit(TOUR_DATE)) &
        (F.col("is_deletion") == False) &
        # validity window: record was active at AS_OF_TIMESTAMP
        (F.col("dbz_timestamp_berlin") <= AS_OF_TIMESTAMP) &
        (F.col("dbz_timestamp_berlin_next").cast("timestamp") > AS_OF_TIMESTAMP)
    )
    .select(
        "tour_id",
        "tour_option_id",
        F.col("date_time_local").alias("slot_time_local"),
        "is_available",
        "vacancies",
        "reserved_vacancies",
        "max_vacancies",
        "availability_type",
        "unavailable_reasons",
        "bookable_until_local",
        "dbz_timestamp_berlin",
    )
    .orderBy("slot_time_local")
)

result = df.limit(200).toPandas()
print(f"tour_id={TOUR_ID}  date={TOUR_DATE}  slots: {len(result)}")
result

SparkException: The query is not executed because it tries to launch 6539 tasks in a single stage, while the maximum allowed tasks one query can launch is 6000; this limit can be modified with configuration parameter "spark.databricks.queryWatchdog.maxQueryTasks".

JVM stacktrace:
org.apache.spark.SparkException
	at com.databricks.spark.util.QueryWatchdog$.checkTooManyTasks(QueryWatchdog.scala:91)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1(QueryWatchdog.scala:76)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1$adapted(QueryWatchdog.scala:71)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at com.databricks.spark.util.QueryWatchdog$.checkNumTasksOfParentStages(QueryWatchdog.scala:71)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1(QueryWatchdog.scala:81)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1$adapted(QueryWatchdog.scal

In [4]:
TOUR_ID      = 1182886      # change me
TOUR_DATE    = "2026-07-15" # change me  (YYYY-MM-DD) — date the tour takes place
BOOKING_DATE = "2026-06-17" # change me  (YYYY-MM-DD) — date the customer would book

# history table is SCD2: each row has a validity window [dbz_timestamp_berlin, dbz_timestamp_berlin_next)
# filtering to that window gives the exact availability state as of BOOKING_DATE
spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "10000")

df = spark.sql(f"""
    SELECT
        tour_id,
        tour_option_id,
        DATE(date_time_local)           AS tour_date,
        date_time_local                 AS slot_time_local,
        is_available,
        vacancies,
        reserved_vacancies,
        max_vacancies,
        availability_type,
        unavailable_reasons,
        bookable_until_local
    FROM production.supply.fact_available_time_slot_history_compacted
    WHERE tour_id               = {TOUR_ID}
      AND DATE(date_time_local) = '{TOUR_DATE}'
      AND is_deletion           = FALSE
      AND dbz_date_berlin       <= DATE('{BOOKING_DATE}')
      AND dbz_timestamp_berlin  <= TIMESTAMP('{BOOKING_DATE} 23:59:59')
      AND CAST(dbz_timestamp_berlin_next AS TIMESTAMP) > TIMESTAMP('{BOOKING_DATE} 00:00:00')
    ORDER BY slot_time_local
""")

df.show(50, truncate=False)

+-------+--------------+---------+---------------+------------+---------+------------------+-------------+-----------------+-------------------+--------------------+
|tour_id|tour_option_id|tour_date|slot_time_local|is_available|vacancies|reserved_vacancies|max_vacancies|availability_type|unavailable_reasons|bookable_until_local|
+-------+--------------+---------+---------------+------------+---------+------------------+-------------+-----------------+-------------------+--------------------+
+-------+--------------+---------+---------------+------------+---------+------------------+-------------+-----------------+-------------------+--------------------+



Some text

In [1]:
spark

In [10]:

# Point-in-time correct bookings for a tour on a given tour_date, as of booking_date
# Uses gyg__booking_history (Debezium CDC) — one row per change event;
# we take the latest state per booking_id before booking_date cutoff.

TOUR_ID      = 62214
TOUR_DATE    = "2024-08-20"
BOOKING_DATE = "2024-07-20"  # the point-in-time: "what did we know on this date?"

pit_sql = f"""
WITH tour_option_ids AS (
    SELECT DISTINCT tour_option_id
    FROM production.dwh.fact_booking
    WHERE tour_id = {TOUR_ID}
),
history_before_date AS (
    SELECT
        h.*,
        ROW_NUMBER() OVER (PARTITION BY h.booking_id ORDER BY h.dbz_timestamp DESC) AS rn
    FROM production.db_mirror_dbz.gyg__booking_history h
    JOIN tour_option_ids t ON h.tour_option_id = t.tour_option_id
    WHERE DATE(h.tour_start_datetime) = '{TOUR_DATE}'
      AND h.dbz_timestamp            <= TIMESTAMP('{BOOKING_DATE} 23:59:59')
)
SELECT
    booking_id,
    tour_option_id,
    tour_start_datetime,
    status,
    date_of_cancelation,
    total_number_of_participants,
    customer_total_price,
    fraud,
    operation,
    dbz_timestamp            AS last_change_before_booking_date
FROM history_before_date
WHERE rn = 1
  AND operation != 'd'
ORDER BY booking_id
"""

df_pit = spark.sql(pit_sql).toPandas()

total       = len(df_pit)
active      = len(df_pit[df_pit["date_of_cancelation"].isna() & (df_pit["fraud"] == False)])
cancelled   = df_pit["date_of_cancelation"].notna().sum()
fraud_count = (df_pit["fraud"] == True).sum()

print(f"tour_id={TOUR_ID}  tour_date={TOUR_DATE}  as_of={BOOKING_DATE}")
print(f"  Total booking rows : {total}")
print(f"  Active (non-cancelled, non-fraud): {active}")
print(f"  Cancelled          : {cancelled}")
print(f"  Fraud              : {fraud_count}")
df_pit.head(10)


tour_id=62214  tour_date=2024-08-20  as_of=2024-07-20
  Total booking rows : 1242
  Active (non-cancelled, non-fraud): 1237
  Cancelled          : 5
  Fraud              : 0


,booking_id,tour_option_id,tour_start_datetime,status,date_of_cancelation,total_number_of_participants,customer_total_price,fraud,operation,last_change_before_booking_date
0,247326781,963214,2024-08-20 10:00:00,active,NaT,1,41.57,False,u,2024-01-06 16:14:38.959
1,247326845,963214,2024-08-20 10:00:00,active,NaT,1,41.57,False,u,2024-01-06 16:14:41.641
2,247416850,963208,2024-08-20 11:30:00,active,NaT,4,124.00,False,u,2024-01-07 13:36:48.653
3,249665388,963214,2024-08-20 11:30:00,active,NaT,2,56.41,False,u,2024-01-28 22:05:40.493
4,250440101,963214,2024-08-20 09:30:00,active,NaT,4,152.00,False,u,2024-02-04 17:22:08.482
5,251001910,963208,2024-08-20 08:00:00,active,NaT,6,205.98,False,u,2024-02-09 03:53:42.273
6,251528600,963208,2024-08-20 11:00:00,active,NaT,3,93.00,False,u,2024-02-13 00:35:30.431
7,251566332,963214,2024-08-20 10:00:00,active,NaT,4,112.62,False,u,2024-02-13 11:05:46.287
8,252455367,963208,2024-08-20 11:30:00,active,NaT,5,113.83,False,u,2024-02-19 23:54:37.849
9,252711983,963214,2024-08-20 16:30:00,active,NaT,4,132.00,False,u,2024-02-22 00:44:42.121


In [11]:

# Validation: compare PIT result against fact_booking (current/final state)
# Expectation:
#   - PIT active count >= fact_booking active count (some bookings active on 2024-07-20 may have cancelled later)
#   - PIT total <= fact_booking total (new bookings made after 2024-07-20 not in PIT)

val_sql = f"""
SELECT
    COUNT(*)                                                              AS total_bookings,
    COUNT(CASE WHEN date_of_cancelation IS NULL AND is_fraud=false THEN 1 END) AS active_bookings,
    COUNT(CASE WHEN date_of_cancelation IS NOT NULL THEN 1 END)          AS cancelled_bookings,
    COUNT(CASE WHEN is_fraud=true THEN 1 END)                            AS fraud_bookings
FROM production.dwh.fact_booking
WHERE tour_id  = {TOUR_ID}
  AND DATE(date_of_travel) = '{TOUR_DATE}'
"""

df_val = spark.sql(val_sql).toPandas()
print("=== fact_booking (CURRENT state) ===")
print(df_val.to_string(index=False))

print()
print("=== PIT history (as of 2024-07-20) ===")
print(f"  total={total}  active={active}  cancelled={cancelled}  fraud={fraud_count}")

print()
print("=== Sanity checks ===")
fb_total    = df_val["total_bookings"].iloc[0]
fb_active   = df_val["active_bookings"].iloc[0]
print(f"  PIT total ({total}) <= fact_booking total ({fb_total}): {total <= fb_total}  ← new bookings made after 2024-07-20")
print(f"  PIT active ({active}) >= fact_booking active ({fb_active}): {active >= fb_active}  ← some bookings active on 2024-07-20 may have cancelled later")


=== fact_booking (CURRENT state) ===
 total_bookings  active_bookings  cancelled_bookings  fraud_bookings
            452              445                   7               0

=== PIT history (as of 2024-07-20) ===
  total=1242  active=1237  cancelled=5  fraud=0

=== Sanity checks ===
  PIT total (1242) <= fact_booking total (452): False  ← new bookings made after 2024-07-20
  PIT active (1237) >= fact_booking active (445): True  ← some bookings active on 2024-07-20 may have cancelled later


In [12]:
spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")

'15.4.x-scala2.12'

In [13]:

# Same as cell 26 but using QUALIFY instead of CTE+WHERE rn=1
TOUR_ID      = 62214
TOUR_DATE    = "2024-08-20"
BOOKING_DATE = "2024-07-20"

pit_sql = f"""
WITH tour_option_ids AS (
    SELECT DISTINCT tour_option_id
    FROM production.dwh.fact_booking
    WHERE tour_id = {TOUR_ID}
)
SELECT
    h.booking_id,
    h.tour_option_id,
    h.tour_start_datetime,
    h.status,
    h.date_of_cancelation,
    h.total_number_of_participants,
    h.customer_total_price,
    h.fraud,
    h.operation,
    h.dbz_timestamp AS last_change_before_booking_date
FROM production.db_mirror_dbz.gyg__booking_history h
JOIN tour_option_ids t ON h.tour_option_id = t.tour_option_id
WHERE DATE(h.tour_start_datetime) = '{TOUR_DATE}'
  AND h.dbz_timestamp            <= TIMESTAMP('{BOOKING_DATE} 23:59:59')
  AND h.operation                != 'd'
QUALIFY ROW_NUMBER() OVER (PARTITION BY h.booking_id ORDER BY h.dbz_timestamp DESC) = 1
ORDER BY h.booking_id
"""

df_pit2 = spark.sql(pit_sql).toPandas()

total       = len(df_pit2)
active      = len(df_pit2[df_pit2["date_of_cancelation"].isna() & (df_pit2["fraud"] == False)])
cancelled   = df_pit2["date_of_cancelation"].notna().sum()
fraud_count = (df_pit2["fraud"] == True).sum()

print(f"tour_id={TOUR_ID}  tour_date={TOUR_DATE}  as_of={BOOKING_DATE}")
print(f"  Total booking rows : {total}")
print(f"  Active (non-cancelled, non-fraud): {active}")
print(f"  Cancelled          : {cancelled}")
print(f"  Fraud              : {fraud_count}")
df_pit2.head(10)


tour_id=62214  tour_date=2024-08-20  as_of=2024-07-20
  Total booking rows : 1303
  Active (non-cancelled, non-fraud): 1298
  Cancelled          : 5
  Fraud              : 0


,booking_id,tour_option_id,tour_start_datetime,status,date_of_cancelation,total_number_of_participants,customer_total_price,fraud,operation,last_change_before_booking_date
0,247326781,963214,2024-08-20 10:00:00,active,NaT,1,41.57,False,u,2024-01-06 16:14:38.959
1,247326845,963214,2024-08-20 10:00:00,active,NaT,1,41.57,False,u,2024-01-06 16:14:41.641
2,247416850,963208,2024-08-20 11:30:00,active,NaT,4,124.00,False,u,2024-01-07 13:36:48.653
3,247766173,963208,2024-08-20 10:00:00,deleted_by_daemon,NaT,2,62.00,False,u,2024-01-10 20:10:08.790
4,247857785,963208,2024-08-20 11:00:00,deleted_by_daemon,NaT,1,26.70,False,u,2024-01-11 18:59:12.766
5,248300406,963208,2024-08-20 12:00:00,deleted_by_daemon,NaT,2,62.00,False,u,2024-01-15 21:33:12.289
6,249188116,963208,2024-08-20 13:00:00,deleted_by_customer,NaT,2,102.28,False,u,2024-01-24 14:31:47.356
7,249319364,963208,2024-08-20 14:00:00,deleted_by_daemon,NaT,5,133.00,False,u,2024-01-25 20:13:02.732
8,249369714,963208,2024-08-20 13:00:00,deleted_by_customer,NaT,2,102.14,False,u,2024-01-26 09:57:43.119
9,249571302,963208,2024-08-20 10:00:00,deleted_by_daemon,NaT,3,82.00,False,u,2024-01-28 08:49:07.125


In [14]:
df = spark.read.json("/Workspace/Users/sergey.rubtsovenko@getyourguide.com/projects/curation-ai/data/synthetic_tour_object.jsonl")
df.printSchema()
print(f"rows: {df.count()}")
df.limit(2).toPandas()

/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2179: UserWarning: Spark Connect Session expired on the server. Please generate a new session by detaching and reattaching the compute if in a Databricks notebook or job or by calling DatabricksSession.builder.getOrCreate() if using Databricks Connect.
  warnings.warn(


SparkConnectGrpcException: (org.apache.spark.SparkSQLException) [INVALID_HANDLE.SESSION_CLOSED] The handle 4c71184d-8396-48b2-803b-81787233869c is invalid. Session was closed. SQLSTATE: HY000

In [16]:
df = spark.read.json("file:/Workspace/Users/sergey.rubtsovenko@getyourguide.com/projects/curation-ai/data/synthetic_tour_object.jsonl")
df.printSchema()
print(f"rows: {df.count()}")
df.limit(2).toPandas()

SparkConnectGrpcException: (org.apache.spark.SparkSQLException) [INVALID_HANDLE.SESSION_CLOSED] The handle 4c71184d-8396-48b2-803b-81787233869c is invalid. Session was closed. SQLSTATE: HY000